In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub
import os
from tqdm import tqdm

csv_path = os.path.join(path, "/kaggle/input/q1-ka-ai-2026/Q1_data.csv")

df = pd.read_csv(csv_path)



In [ ]:
# Task 2: Write your code here:
df.head()


In [ ]:
# Task 3: Write your code here:
df.info()


In [ ]:
# Task 4: Write your code here:
df.describe()


In [ ]:
# Task 5: Write your code here:
def check_target_distribution(df, target_column):
  df[target_column].hist(bins=30, edgecolor='black')

  plt.title(f"Target Distribution ({target_column})")
  plt.xlabel(target_column)
  plt.ylabel("Frequency")
  plt.grid(False)

  plt.show()

check_target_distribution(df, "Delivery_Time")

In [ ]:
# Task 1: Write your code here:
df=df.drop(columns=['Order_ID'])
df.info()

In [ ]:
# Task 2: Write your code here:
def check_missing_values(df):

  # Get missing values using pandas
  missing_values = df.isnull().sum()

  print("Missing Values per Column:")
  print(missing_values[missing_values > 0])

  if missing_values.any():
    print("\nHandle Missing Values as needed.")
  else:
    print("\nNo Missing Values Found.")


df=df.dropna()
check_missing_values(df)

In [ ]:
# Task 3: Write your code here:
def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df)

In [ ]:
# Task 4: Write your code here:
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
categorical_cols = df.select_dtypes(include=["object"]).columns
onehot_encoder = OneHotEncoder(sparse_output=False)
df[categorical_cols]= le.fit_transform(categorical_cols)
df.info()
df.head()

In [ ]:
# Task 5: Write your code here:
from sklearn.preprocessing import StandardScaler
standard_scaler = StandardScaler()
data_standard_scaled = standard_scaler.fit_transform(df)

In [ ]:
# Task 6: Write your code here:
def check_target_imbalance(df, target_column):
  print("Target Distribution:")

  df[target_column].hist()  # Yeah you can just do this :)
  plt.show()

check_target_imbalance(df, "Delivery_Time")
#it's impalance

In [ ]:
# Task 1: Write your code here:
X = df.drop("Delivery_Time", axis=1).astype(float)
y = df['Delivery_Time'].astype(float)

In [ ]:
# Task 2,3,4,5: Write your code here:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import  mean_absolute_error as mae1
model=RandomForestRegressor(n_estimators=200)


from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score
n_splits = 5 # K=5 Folds
lr_mae = []
# Stratified 5-Fold Cross-Validation, shuffled
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
for fold_idx, (train_index, test_index) in enumerate(skf.split(X,y)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")

  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  # Train
  model.fit(X_train, y_train)
  y_pred = model.predict(X_test)
  # Validate


  mae=mae1(y_test, y_pred)
  lr_mae.append(mae)
np.mean(lr_mae)


In [ ]:
# Task 1: Write your code here:
# Feature importance
importances =model.feature_importances_
features = X.columns
feature_importance = pd.DataFrame({
    'features': features,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(feature_importance['features'], feature_importance['importance'])
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Task 2: Write your code here:
df['Delivery_Time'].hist(bins=30, edgecolor='black')
plt.title(f"Target Distribution ({'Delivery_Time'})")
plt.xlabel('target_column')
plt.ylabel("Frequency")
plt.grid(False)
plt.show()

In [ ]:
!pip install catboost

In [ ]:
# Task Bonus: Write your code here:

from sklearn.ensemble import RandomForestRegressor
from catboost import CatBoostRegressor
from sklearn.metrics import  mean_absolute_error as mae1
models = {
  "Random Forest Regressor": RandomForestRegressor(n_estimators=200),
  "CatBoost": CatBoostRegressor(verbose=0)
}

all_results = {}

for name in models:
  all_results[name] = {'mae':[]}

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import  f1_score
n_splits = 5 # K=5 Folds
lr_mae = []
# Stratified 5-Fold Cross-Validation, shuffled
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
for fold_idx, (train_index, test_index) in enumerate(skf.split(X,y)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")

  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]
  for model_name, model in models.items():
    print(f"Training {model_name}...")
  # Train
    model.fit(X_train, y_train)
    #prediction of both models
    y_pred = model.predict(X_test)
    #calculating MAE
    mae=mae1(y_test, y_pred)

    all_results[model_name]["mae"].append(mae)


for model_name in all_results:
  print(f"\n{model_name}:")
  print(f"  MAE:  {np.mean(all_results[model_name]['mae']):.4f}")